In [1]:
import io, pickle
import numpy as np

dodaj train_data.zip a test_data.csv

# Trace Twins — BASELINE notebook

Run top to bottom. Replace the `Submission` below's `score_A`/`score_B` with your real methods**, then build and submit `submission.pkl`.

### 1. Setup (unzip the train data)

In [2]:
# The train data is in this notebook's folder as train_data.zip — unzip it.
!unzip -o train_data.zip          # -> public_traces.csv

Archive:  train_data.zip
  inflating: public_traces.csv       


In [3]:
import pandas as pd
df = pd.read_csv("public_traces.csv")
print(df.head())

   program_id category                                             tokens
0        5080   Adware  ldrloaddll ldrgetprocedureaddress ldrgetproced...
1        4607   Adware  ldrloaddll ldrgetprocedureaddress ldrgetproced...
2        2264   Adware  ntallocatevirtualmemory ntfreevirtualmemory nt...
3        3941   Adware  ldrloaddll ldrgetprocedureaddress ldrgetproced...
4        4997   Adware  ldrloaddll ldrgetprocedureaddress ldrgetproced...


In [4]:
len(df)

4843

In [5]:
all_tokens = set()
for words in df["tokens"].str.split():
    all_tokens.update(words)
num_tokens = len(all_tokens)
print(f"Number of tokens: {num_tokens}")

Number of tokens: 275


In [6]:
token_labels = dict(zip(all_tokens, range(num_tokens)))

### 2. Your `Submission` (edit `score_A`/`score_B`)

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [8]:
class SiameseNetwork(nn.Module):
    def __init__(self, num_tokens, d_model=16, nhead=1, num_layers=3):
        super(SiameseNetwork, self).__init__()
        self.embedding = nn.Embedding(num_tokens, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(1, num_tokens, d_model) )
        self.cls = nn.Parameter(torch.randn(1,1,d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, enable_nested_tensor=False)
    def forward(self, x):
        x = self.embedding(x)
        x = x + self.pos_embedding[:, :x.size(1), :]
        cls_token = self.cls.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = self.transformer_encoder(x)
        x = x[:, 0, :]
        return x

In [9]:
class Submission:
    def __init__(self):
        # Load or train your models here. The baseline currently needs nothing.
        self.model_A = SiameseNetwork(num_tokens,16,1,3)
        self.model_B = SiameseNetwork(num_tokens,16,1,3)

    @staticmethod
    def tokenize_A(window):
        pattern = []
        for word in window:
            pattern.append(token_labels[word])
        return torch.tensor(pattern)

    @staticmethod
    def tokenize_B(window):
        labels = {}
        pattern = []
        for word in window:
            if word not in labels:
                labels[word] = len(labels)
            pattern.append(labels[word])
        return torch.tensor(pattern)

    # ===== DO NOT EDIT: the cloud calls this to collect your scores =====
    def __call__(self, data: bytes) -> bytes:
        req = pickle.loads(data)
        fn = self.score_A if req["part"] == "A" else self.score_B
        scores = fn(req["windows"], req["pairs"])
        buf = io.BytesIO(); np.save(buf, np.asarray(list(scores), dtype=np.float64))
        return buf.getvalue()
    # ===================================================================

    def score_A(self, windows, pairs):
        tokenids = torch.stack([Submission.tokenize_A(window) for window in windows])
        embeddings = self.model_A(tokenids)
        pairs_t = torch.tensor(pairs)
        return [ score.item() for score in F.cosine_similarity(embeddings[pairs_t[:,0]], embeddings[pairs_t[:,1]]) ]

    def score_B(self, windows, pairs):
        tokenids = torch.stack([Submission.tokenize_B(window) for window in windows])
        embeddings = self.model_B(tokenids)
        pairs_t = torch.tensor(pairs)
        return [ score.item() for score in F.cosine_similarity(embeddings[pairs_t[:,0]], embeddings[pairs_t[:,1]]) ]

### 3. Train your solution (run once)

In [10]:
sol = Submission()   # loads/trains everything

In [11]:
def train(model, tokenizer, iters=10):
    model.to('cuda')
    model.train()
    criterion = nn.CosineEmbeddingLoss(margin=0.5)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    num = 20
    for it in range(iters):
        i = np.random.randint(len(df))
        j = np.random.randint(len(df))
        if i == j:
            continue
        row1 = df.iloc[i]
        row2 = df.iloc[j]
        id1 = row1["program_id"]
        id2 = row2["program_id"]
        if id1 == id2:
            continue
        tokens1 = row1["tokens"].split()
        tokens2 = row2["tokens"].split()
        if len(tokens1) < 200 or len(tokens2) < 200:
            continue

        idx1 = np.random.permutation(len(tokens1)-200+1)[:num]
        idx2 = np.random.permutation(len(tokens2)-200+1)[:num]

        token_ids1 = torch.stack([tokenizer(tokens1[idx:idx+200]) for idx in idx1])
        token_ids2 = torch.stack([tokenizer(tokens2[idx:idx+200]) for idx in idx2])

        x = torch.cat([token_ids1, token_ids2],dim=0)
        x = x.cuda()
        y = model(x)

        label = torch.tensor([id1]*len(idx1) + [id2]*len(idx2),dtype=torch.float32)
        label = label.cuda()

        y1 = y.repeat(y.shape[0], 1)
        y2 = y[:,None].repeat(1,y.shape[0],1).reshape(-1,y.shape[1])

        label1 = label.repeat(label.shape[0])
        label2 = label[:,None].repeat(1,label.shape[0]).reshape(-1)

        target = torch.where(label1 == label2, 1, -1)
        loss = criterion(y1, y2, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if it % 100 == 0:
            print(it, loss.item())

    model.eval()
    model.to('cpu')

In [12]:
train(sol.model_A,sol.tokenize_A,20000)

0 0.23596470057964325
100 0.1081322580575943
200 0.13068930804729462
300 0.15849190950393677
400 0.04848519340157509
600 0.02696988917887211
900 0.240103080868721
1000 0.20948375761508942
1100 0.1939326375722885
1200 0.07511827349662781
1300 0.0351903960108757
1400 0.1865609884262085
1500 0.22948725521564484
1600 0.2071557641029358
1800 0.22123514115810394
1900 0.08495459705591202
2000 0.13389182090759277
2200 0.14338763058185577
2300 0.22146813571453094
2400 0.052385080605745316
2500 0.09658490121364594
2600 0.021109363064169884
2800 0.23364484310150146
2900 0.034930966794490814
3000 0.17543840408325195
3200 0.16284076869487762
3300 0.10397141426801682
3400 0.08959277719259262
3500 0.066098652780056
3600 0.173334538936615
3800 0.031345367431640625
3900 0.18083716928958893
4000 0.14825177192687988
4100 0.1363813877105713
4300 0.20279425382614136
4400 0.1738443374633789
4500 0.09376442432403564
4600 0.13148048520088196
4700 0.039257559925317764
4800 0.044144801795482635
4900 0.025169057

In [13]:
train(sol.model_B,sol.tokenize_B,30000)

100 0.20426471531391144
200 0.2159816473722458
400 0.23169374465942383
500 0.23298518359661102
600 0.2222381979227066
900 0.06524863094091415
1000 0.272047221660614
1100 0.12197130173444748
1200 0.2594246566295624
1300 0.18219776451587677
1400 0.02476057969033718
1500 0.19333095848560333
1600 0.17819106578826904
1700 0.14441102743148804
1800 0.2388879358768463
2000 0.07090988010168076
2100 0.17223739624023438
2200 0.23294873535633087
2300 0.19055917859077454
2400 0.23412875831127167
2500 0.22793571650981903
2600 0.15878361463546753
2700 0.13260102272033691
2800 0.22031715512275696
2900 0.2425471842288971
3000 0.24189428985118866
3100 0.12332333624362946
3200 0.24822044372558594
3300 0.24612130224704742
3500 0.08772528171539307
3600 0.2346523404121399
3800 0.050938889384269714
3900 0.22779323160648346
4000 0.24324461817741394
4100 0.10726089030504227
4200 0.12734706699848175
4300 0.24167144298553467
4400 0.25317850708961487
4500 0.21945735812187195
4600 0.13110169768333435
4700 0.006512

### 4. (optional) Estimate your score locally

In [14]:
# OPTIONAL local check — self-contained; mirrors the grader (disjoint Part A / Part B
# programs, per-window scramble for B, 50+50 bands). Needs only public_traces.csv.
import csv, random
import numpy as np
from collections import defaultdict
from sklearn.metrics import roc_auc_score

WINDOW = 200; WPP = 8
def _load(p):
    out=[]
    with open(p) as f:
        r=csv.reader(f); next(r)
        for pid,cat,toks in r: out.append({"program_id":int(pid),"category":cat,"tokens":toks.split()})
    return out
def _windows(traces):
    out=[]
    for tr in traces:
        s=tr["tokens"]; n=(len(s)//WINDOW)*WINDOW
        out.extend([{"program_id":tr["program_id"],"category":tr["category"],"wid":j//WINDOW,
                     "tokens":s[j:j+WINDOW]} for j in range(0,n,WINDOW)][:WPP])
    return out
def _pairs(ws,n,seed):
    rng=random.Random(seed); bc=defaultdict(list); bp=defaultdict(list)
    for k,w in enumerate(ws): bc[w["category"]].append(k); bp[w["program_id"]].append(k)
    cats=sorted(bc); multi=[p for p in bp if len(bp[p])>=2]; P=[]; L=[]
    while len(P)<n:
        if rng.random()<0.5:
            p=rng.choice(multi); a,b=rng.sample(bp[p],2); P.append((a,b)); L.append(1)
        else:
            pool=bc[rng.choice(cats)]
            for _ in range(50):
                a,b=rng.sample(pool,2)
                if ws[a]["program_id"]!=ws[b]["program_id"]: P.append((a,b)); L.append(0); break
    return P,L
def _scramble(ws,off):
    vocab=sorted({t for w in ws for t in w["tokens"]}); out=[]
    for w in ws:
        r=random.Random((w["program_id"]*1_000_000+w["wid"])^off); sh=list(vocab); r.shuffle(sh)
        m=dict(zip(vocab,sh)); out.append([m[t] for t in w["tokens"]])
    return out

tr=_load("public_traces.csv")
by_prog={}
for t in tr: by_prog.setdefault(t["program_id"], t)
ids=sorted(by_prog); random.Random(7).shuffle(ids)
val=ids[:int(len(ids)*0.30)]; h=len(val)//2
wA=_windows([by_prog[i] for i in val[:h]]); WA=[w["tokens"] for w in wA]; pA,lA=_pairs(wA,3000,101)
wB=_windows([by_prog[i] for i in val[h:]]); WB=_scramble(wB,303); pB,lB=_pairs(wB,3000,202)
pts=lambda a,b: max(0.0, min(50.0,(a-0.5)/b*50.0))
aucA=roc_auc_score(lA, sol.score_A(WA,pA)); aucB=roc_auc_score(lB, sol.score_B(WB,pB))
print(f"Part A: AUC {aucA:.3f} -> {pts(aucA,0.34):.1f}/50")
print(f"Part B: AUC {aucB:.3f} -> {pts(aucB,0.28):.1f}/50")
print(f"ESTIMATED TOTAL ~ {pts(aucA,0.34)+pts(aucB,0.28):.1f}/100  (secret set differs slightly)")

Part A: AUC 0.797 -> 43.7/50
Part B: AUC 0.647 -> 26.2/50
ESTIMATED TOTAL ~ 69.9/100  (secret set differs slightly)


### 5. Build submission.pkl  (run LAST)

In [15]:
# Build submission.pkl  (this is what you upload as the Output)
import cloudpickle, os
# `sol` was trained above. Re-run the train cell first if you restarted the kernel.
with open("submission.pkl", "wb") as f:
    cloudpickle.dump(sol, f)
mb = os.path.getsize("submission.pkl") / 1e6
print(f"wrote submission.pkl  ({mb:.1f} MB)  -- must be < 50 MB")
assert mb < 50, "too big: cap model size (fewer trees / depth)"

wrote submission.pkl  (1.8 MB)  -- must be < 50 MB


In [ ]:
mb

36.331198

### 6. Submit
Click the **🦆 Submit to Judge** button in the toolbar and choose `submission.pkl` as the Output.